<a href="https://colab.research.google.com/github/Rist-A/Icog_Project_1_EV_Imports/blob/main/notebooks/Wubrist_Alemu_01_clean_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_FOLDER = "vehicle_parts_project"      # the folder name in your Google Drive (Colab only)

def find_root():
    # 1) running as a file inside the project (python run_all.py): the folder above notebooks/
    try:
        here = Path(__file__).resolve().parent
        if here.name == "notebooks":
            return here.parent
    except NameError:
        pass
    # 2) Google Colab: MyDrive/<PROJECT_FOLDER> (asks for Drive permission once)
    drive_root = Path("/content/drive/MyDrive")
    try:
        from google.colab import drive
        if not drive_root.exists():
            drive.mount("/content/drive")
    except Exception:
        pass
    if drive_root.exists():
        return drive_root / PROJECT_FOLDER
    # 3) anywhere else: the current folder (or its parent if you are inside notebooks/)
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd

ROOT = find_root()

RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

VEHICLES = ["BYD Seagull", "BYD Yuan Up", "Volkswagen ID.4/ID.6"]
PARTS = ["Brake pads", "Brake discs", "Cabin-air filters", "Wiper blades", "Headlights",
         "Tail lights", "Side mirrors", "Bumpers", "Fenders/body panels", "Shock absorbers"]
COMPAT_STATUSES = ["confirmed_2_sources", "listed_in_2_sources_codes_differ", "confirmed_1_source", "possible_fit", "not_found"]
NUMERIC = ["n_catalogue_matches", "price_original", "fx_rate_to_usd", "price_usd", "price_on_request", "in_stock",
           "wear_item", "nhtsa_available", "nhtsa_complaints", "local_sellers", "n_sources"]
BINARY = ["in_stock", "wear_item", "price_on_request", "nhtsa_available"]
KEY_EVIDENCE = ["oem_number", "price_original", "in_stock", "wear_item", "nhtsa_complaints", "local_sellers"]

log = []
def note(step, detail, n=None):
    log.append({"step": step, "detail": detail, "rows_affected": n})
    print(f"[{step}] {detail}" + (f" ({n})" if n is not None else ""))

## 1. Load

In [ ]:
src = RAW / "vehicle_part_combinations.csv"
if not src.exists():
    src = RAW / "vehicle_part_combinations_scraped.csv"
    print("WARNING: vehicle_part_combinations.csv not found - using the scraped sheet.")
    print("         wear_item and local_sellers are still blank until you fill them in.")
assert src.exists(), "Run 00_collect_evidence first (it creates vehicle_part_combinations_scraped.csv)"
df = pd.read_csv(src, dtype=str, keep_default_na=False)
for c in df.columns:
    df[c] = df[c].str.strip()
print("Loaded", src.name, "-", len(df), "rows")

         wear_item and local_sellers are still blank until you fill them in.
Loaded vehicle_part_combinations_scraped.csv - 30 rows


## 2. Names, duplicates, numbers and text checks

In [ ]:
# 1. names, duplicates
bad = ~df["vehicle"].isin(VEHICLES) | ~df["part"].isin(PARTS)
if bad.any():
    note("names", "rows with an unknown vehicle or part name dropped", int(bad.sum()))
    df = df[~bad]
dup = df.duplicated(["vehicle", "part"], keep="first")
if dup.any():
    note("duplicates", "duplicate vehicle-part rows dropped (first kept)", int(dup.sum()))
    df = df[~dup]
missing_rows = len(VEHICLES) * len(PARTS) - len(df)
if missing_rows:
    note("coverage", "vehicle-part combinations absent from the sheet", missing_rows)

# 2. numbers
for c in NUMERIC:
    if c in df:
        nonblank = df[c] != ""
        as_num = pd.to_numeric(df[c], errors="coerce")
        if (nonblank & as_num.isna()).any():
            note("numbers", f"{c}: non-numeric text turned into blank", int((nonblank & as_num.isna()).sum()))
        df[c] = as_num
for c in BINARY:
    wrong = df[c].notna() & ~df[c].isin([0, 1])
    if wrong.any():
        note("numbers", f"{c}: values other than 0/1 set to blank", int(wrong.sum()))
        df.loc[wrong, c] = np.nan
neg = (df["price_original"] < 0) | (df["local_sellers"] < 0) | (df["nhtsa_complaints"] < 0)
if neg.any():
    note("numbers", "negative values set to blank", int(neg.sum()))
    df.loc[df["price_original"] < 0, "price_original"] = np.nan
    df.loc[df["local_sellers"] < 0, "local_sellers"] = np.nan
    df.loc[df["nhtsa_complaints"] < 0, "nhtsa_complaints"] = np.nan

# 3. text checks
bad_status = (df["compat_status"] != "") & ~df["compat_status"].isin(COMPAT_STATUSES)
if bad_status.any():
    note("text", "unknown compat_status set to not_found", int(bad_status.sum()))
df.loc[bad_status | (df["compat_status"] == ""), "compat_status"] = "not_found"

sl = RAW / "source_log.csv"
if sl.exists():
    known = set(pd.read_csv(sl, dtype=str)["source_id"])
    used = set(";".join(df["compat_source_ids"]).split(";")) | set(df["price_source_id"]) - {""}
    unknown = {u for u in used if u and u not in known}
    if unknown:
        note("sources", f"source ids not in source_log.csv: {sorted(unknown)}", None)
else:
    note("sources", "source_log.csv not found - run 00_collect_evidence", None)

## 3. Currency and judgement columns

In [ ]:
# 4. currency: USD only when an FX rate and its date are given
cur = set(df.loc[df["price_original"].notna(), "currency_original"]) - {""}
need_usd = df["price_usd"].isna() & df["price_original"].notna() & df["fx_rate_to_usd"].notna()
df.loc[need_usd, "price_usd"] = (df["price_original"] * df["fx_rate_to_usd"]).round(2)
no_date = df["fx_rate_to_usd"].notna() & (df["fx_rate_date"] == "")
if no_date.any():
    note("currency", "FX rate given without an FX date - add the date", int(no_date.sum()))
if len(cur) > 1 and df["price_usd"].isna().any():
    note("currency", f"prices in several currencies {sorted(cur)} and some have no USD value - fix before scoring", None)
note("currency", f"price currencies found: {sorted(cur) or 'none'}; USD calculated for {int(need_usd.sum())} rows")

# 5. judgement columns must be separate from observed facts
no_basis = df["wear_item"].notna() & (df["wear_basis"] == "")
if no_basis.any():
    note("judgement", "wear_item filled without a wear_basis (reason) - add the reason", int(no_basis.sum()))
note("judgement", "wear_item is analyst judgement; blank in this many rows", int(df["wear_item"].isna().sum()))
note("judgement", "local_sellers is your own check; blank in this many rows", int(df["local_sellers"].isna().sum()))

# 6. missing report
def blank(row, c):
    v = row[c]
    return v == "" or (isinstance(v, float) and np.isnan(v))
df["missing_fields"] = df.apply(lambda r: ";".join(c for c in KEY_EVIDENCE if blank(r, c)), axis=1)
df["n_missing_key"] = df["missing_fields"].apply(lambda s: len([x for x in s.split(";") if x]))

[currency] price currencies found: ['RMB']; USD calculated for 0 rows
[judgement] wear_item is analyst judgement; blank in this many rows (30)
[judgement] local_sellers is your own check; blank in this many rows (30)


## 4. Save and report what is missing

In [ ]:
cleaned = df.sort_values(["vehicle", "part"]).reset_index(drop=True)
cleaned.to_csv(PROC / "cleaned_dataset.csv", index=False)

miss = pd.DataFrame({"missing_cells": [int(cleaned[c].apply(lambda v: v == "" or (isinstance(v, float) and np.isnan(v))).sum()) for c in KEY_EVIDENCE]},
                    index=KEY_EVIDENCE)
miss["of_total"] = len(cleaned)
miss["pct_missing"] = (100 * miss["missing_cells"] / len(cleaned)).round(1)
miss.to_csv(PROC / "missing_report.csv")

fitment_ok = ["confirmed_2_sources", "listed_in_2_sources_codes_differ", "confirmed_1_source"]
cov = cleaned.groupby("vehicle").agg(
    parts=("part", "size"),
    fitment_found=("compat_status", lambda s: int(s.isin(fitment_ok).sum())),
    fitment_2_sources=("compat_status", lambda s: int(s.isin(["confirmed_2_sources", "listed_in_2_sources_codes_differ"]).sum())),
    price_found=("price_original", lambda s: int(s.notna().sum())),
    nhtsa_rows=("nhtsa_complaints", lambda s: int(s.notna().sum())),
    wear_filled=("wear_item", lambda s: int(s.notna().sum())),
)
cov.to_csv(PROC / "coverage_by_vehicle.csv")
pd.DataFrame(log).to_csv(PROC / "cleaning_log.csv", index=False)
print("Saved cleaned_dataset.csv,", len(cleaned), "rows")
print("\nMissing evidence:"); print(miss)
print("\nCoverage by vehicle:"); cov

Saved cleaned_dataset.csv, 30 rows

Missing evidence:
                  missing_cells  of_total  pct_missing
oem_number                   21        30         70.0
price_original               22        30         73.3
in_stock                     30        30        100.0
wear_item                    30        30        100.0
nhtsa_complaints             20        30         66.7
local_sellers                30        30        100.0

Coverage by vehicle:


,parts,fitment_found,fitment_2_sources,price_found,nhtsa_rows,wear_filled
vehicle,,,,,,
BYD Seagull,10,9,9,8,0,0
BYD Yuan Up,10,0,0,0,0,0
Volkswagen ID.4/ID.6,10,0,0,0,10,0
